# Recalculate velocities

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import pyarrow as pa
import pyarrow.parquet as pq
import gc
import psutil
import os
import geopandas as gpd
import shutil

In [2]:
# ============================================================
# Paths
# ============================================================

INPUT_DIR = (
    Path.home()
    / "projects"
    / "ADUCAT"
    / "data"
    / "EGMS"
    / "vienna"
)

OUTPUT_DIR = INPUT_DIR / "processed"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Temporary chunks
temp_dir = OUTPUT_DIR / "temp_chunks"

temp_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
# ============================================================
# Input settings
# ============================================================

ORBIT = "022"
PRODUCT_START = "2020"
PRODUCT_END = "2024"

# ============================================================
# Find input files
# ============================================================

PRODUCT_RANGE = f"{PRODUCT_START}_{PRODUCT_END}"

csv_files = sorted(
    f for f in INPUT_DIR.glob("*.csv")
    if (
        f"_{ORBIT}_" in f.name
        and f"_{PRODUCT_RANGE}_" in f.name
    )
)

print(
    f"Found {len(csv_files)} CSV files "
    f"for orbit {ORBIT}, "
    f"product period {PRODUCT_RANGE}"
)

for f in csv_files:
    print(f"  {f.name}")

Found 2 CSV files for orbit 022, product period 2020_2024
  EGMS_L2a_022_0787_IW1_VV_2020_2024_1.csv
  EGMS_L2a_022_0788_IW1_VV_2020_2024_1.csv


In [4]:
# Final output
final_output = (
    OUTPUT_DIR /
    f"EGMS_{ORBIT}_{PRODUCT_RANGE}.parquet"
)

print("Final file name:", final_output)

Final file name: /home/lukas/projects/ADUCAT/data/EGMS/vienna/processed/EGMS_022_2020_2024.parquet


### Functions

In [5]:
def add_velocity(gdf, intervals):
    """
    Calculate linear velocity and velocity standard error
    for EGMS time-series data.

    EGMS time-series columns are named YYYYMMDD, e.g.
    20220101, 20220113, ...

    The regression is identical to the Augmenterra implementation:

        np.polyfit(t, y, 1, cov=True)

    Velocity is the fitted slope in mm/year.
    """

    # --------------------------------------------------
    # Find EGMS time-series columns
    # --------------------------------------------------

    ts_cols = [
        c for c in gdf.columns
        if re.fullmatch(r"\d{8}", str(c))
    ]

    if not ts_cols:
        raise ValueError(
            "No EGMS time-series columns found."
        )

    dates = pd.to_datetime(
        ts_cols,
        format="%Y%m%d"
    )

    # --------------------------------------------------
    # Process each interval
    # --------------------------------------------------

    for start_date, end_date in intervals:

        print(
            f"    Processing "
            f"{start_date} → {end_date}"
        )

        start_date = pd.Timestamp(start_date)
        end_date = pd.Timestamp(end_date)

        # --------------------------------------------------
        # Select observations in this interval
        # --------------------------------------------------

        mask = (
            (dates >= start_date) &
            (dates <= end_date)
        )

        selected_cols = [
            c
            for c, keep in zip(ts_cols, mask)
            if keep
        ]

        selected_dates = dates[mask]

        if len(selected_dates) < 3:

            print(
                "      Skipping: "
                "fewer than 3 observations"
            )

            continue

        print(
            f"      Using {len(selected_dates)} observations "
            f"({selected_cols[0]} → {selected_cols[-1]})"
        )

        # --------------------------------------------------
        # Time in years
        # --------------------------------------------------

        t = (
            selected_dates - selected_dates[0]
        ).total_seconds() / (
            365.25 * 24 * 3600
        )

        t = np.asarray(
            t,
            dtype=np.float64
        )

        # --------------------------------------------------
        # Output arrays
        # --------------------------------------------------

        velocities = np.full(
            len(gdf),
            np.nan,
            dtype=np.float64
        )

        velocity_std = np.full(
            len(gdf),
            np.nan,
            dtype=np.float64
        )

        # --------------------------------------------------
        # Process every point in this chunk
        # --------------------------------------------------

        for local_i, (_, row) in enumerate(
            gdf.iterrows()
        ):

            y = pd.to_numeric(
                row[selected_cols],
                errors="coerce"
            ).to_numpy(
                dtype=np.float64
            )

            valid = np.isfinite(y)

            if valid.sum() >= 3:

                try:

                    coefficients, covariance = np.polyfit(
                        t[valid],
                        y[valid],
                        1,
                        cov=True
                    )

                    # --------------------------------------
                    # Mean linear velocity [mm/year]
                    # --------------------------------------

                    velocities[local_i] = round(
                        coefficients[0],
                        1
                    )

                    # --------------------------------------
                    # Standard error of velocity [mm/year]
                    # --------------------------------------

                    velocity_std[local_i] = round(
                        np.sqrt(covariance[0, 0]),
                        1
                    )

                except (np.linalg.LinAlgError, ValueError):
                    pass

            del y
            del valid

        # --------------------------------------------------
        # Add result columns
        # --------------------------------------------------

        start_str = start_date.strftime("%Y%m%d")
        end_str = end_date.strftime("%Y%m%d")

        vel_col = f"VEL_{start_str}_{end_str}"
        std_col = f"V_STDERR_{start_str}_{end_str}"

        new_columns = pd.DataFrame(
            {
                vel_col: velocities,
                std_col: velocity_std
            },
            index=gdf.index
        )

        gdf = pd.concat(
            [gdf, new_columns],
            axis=1
        )

        # --------------------------------------------------
        # Clean interval-specific memory
        # --------------------------------------------------

        del velocities
        del velocity_std
        del selected_cols
        del selected_dates
        del mask
        del t

        gc.collect()

    # --------------------------------------------------
    # Clean function-level objects
    # --------------------------------------------------

    del ts_cols
    del dates

    gc.collect()

    return gdf

In [6]:
# ============================================================
# Determine date columns
# ============================================================

def get_date_columns(csv_file, start_date, end_date):

    columns = pd.read_csv(
        csv_file,
        nrows=0
    ).columns

    date_columns = [
        col for col in columns
        if re.fullmatch(r"\d{8}", str(col))
        and start_date <= str(col) <= end_date
    ]

    return date_columns

In [7]:
process = psutil.Process(os.getpid())


def print_memory(label=""):

    rss_gb = (
        process.memory_info().rss
        / (1024 ** 3)
    )

    print(
        f"[MEMORY] {label}: "
        f"{rss_gb:.2f} GB RSS"
    )

### Processing

In [8]:
intervals = [
    ("2022-01-01", "2024-12-31"),
    ("2020-01-01", "2024-12-31")
]


CHUNK_SIZE = 200000

In [9]:

csv_files = sorted(
    f for f in INPUT_DIR.glob("*.csv")
    if (
        f"_{ORBIT}_" in f.name
        and f"_{PRODUCT_START}_{PRODUCT_END}_" in f.name
    )
)

print(
    f"Found {len(csv_files)} EGMS CSV files "
    f"for orbit {ORBIT}, "
    f"product period {PRODUCT_START}_{PRODUCT_END}"
)


Found 2 EGMS CSV files for orbit 022, product period 2020_2024


In [10]:
# ============================================================
# TEST MODE
# ============================================================

Test = False  # Set to True for testing, False for the full dataset


# ============================================================
# TEST: first and last file only
# ============================================================

if Test:
    if len(csv_files) < 2:
        raise ValueError(
            "Need at least two CSV files for this test."
        )

    csv_files = [
        csv_files[0],
        csv_files[-1]
    ]

    print()
    print(f"Selected {len(csv_files)} CSV files:")
    for f in csv_files:
        print(f"  {f.name}")

else:
    print("Using all CSV files.")

Using all CSV files.


In [11]:
# ============================================================
# PROCESS EGMS CSV FILES
# ============================================================

temp_files = []

global_chunk_number = 0


for f in csv_files:
    print(f"  {f.name}")


# ============================================================
# PROCESS EACH CSV
# ============================================================

for csv_index, csv_file in enumerate(
    csv_files,
    start=1
):

    print()
    print("=" * 70)
    print(
        f"CSV FILE "
        f"{csv_index}/{len(csv_files)}"
    )
    print(csv_file.name)
    print("=" * 70)

    # --------------------------------------------------------
    # Determine number of rows and chunks
    # --------------------------------------------------------

    with open(csv_file, "rb") as f:
        feature_count = sum(1 for _ in f) - 1  # subtract header

    number_of_chunks = (
        feature_count + CHUNK_SIZE - 1
    ) // CHUNK_SIZE

    print(
        f"Total points: "
        f"{feature_count:,}"
    )

    print(
        f"Chunk size: "
        f"{CHUNK_SIZE:,}"
    )

    print(
        f"Chunks to process: "
        f"{number_of_chunks:,}"
    )

    # --------------------------------------------------------
    # Process CSV in chunks
    # --------------------------------------------------------

    for chunk_index, gdf_chunk in enumerate(
        pd.read_csv(
            csv_file,
            chunksize=CHUNK_SIZE
        ),
        start=1
    ):

        print()
        print(
            f"Chunk {chunk_index}"
        )

        print_memory(
            "after reading chunk"
        )

        print(
            f"Loaded: "
            f"{len(gdf_chunk):,} points"
        )

        # ----------------------------------------------------
        # CALCULATE VELOCITIES
        # ----------------------------------------------------

        gdf_chunk = add_velocity(
            gdf_chunk,
            intervals
        )

        print_memory(
            "after velocity calculation"
        )

        # ----------------------------------------------------
        # CREATE GEOMETRY
        # ----------------------------------------------------

        gdf_chunk = gpd.GeoDataFrame(
            gdf_chunk,
            geometry=gpd.points_from_xy(
                gdf_chunk["longitude"],
                gdf_chunk["latitude"]
            ),
            crs="EPSG:4326"
        )

        gdf_chunk = gdf_chunk.drop(
            columns=["longitude", "latitude"]
        )

        print(
            "Geometry created: "
            f"{gdf_chunk.geometry.notna().sum():,} points"
        )

        print_memory(
            "after geometry creation"
        )

        # ----------------------------------------------------
        # WRITE TEMPORARY GEOPARQUET
        # ----------------------------------------------------

        global_chunk_number += 1

        temp_file = (
            temp_dir /
            f"part_{global_chunk_number:05d}.parquet"
        )

        gdf_chunk.to_parquet(
            temp_file,
            index=False,
            compression="zstd"
        )

        temp_files.append(
            temp_file
        )

        print(
            f"Written GeoParquet: "
            f"{temp_file.name}"
        )

        print_memory(
            "after writing chunk"
        )

        # ----------------------------------------------------
        # RELEASE CHUNK
        # ----------------------------------------------------

        del gdf_chunk

        gc.collect()

        print_memory(
            "after chunk cleanup"
        )

  EGMS_L2a_022_0787_IW1_VV_2020_2024_1.csv
  EGMS_L2a_022_0788_IW1_VV_2020_2024_1.csv

CSV FILE 1/2
EGMS_L2a_022_0787_IW1_VV_2020_2024_1.csv
Total points: 513,591
Chunk size: 200,000
Chunks to process: 3

Chunk 1
[MEMORY] after reading chunk: 0.60 GB RSS
Loaded: 200,000 points
    Processing 2022-01-01 → 2024-12-31
      Using 88 observations (20220110 → 20241225)
    Processing 2020-01-01 → 2024-12-31
      Using 207 observations (20200103 → 20241225)
[MEMORY] after velocity calculation: 0.71 GB RSS
Geometry created: 200,000 points
[MEMORY] after geometry creation: 1.06 GB RSS
Written GeoParquet: part_00001.parquet
[MEMORY] after writing chunk: 1.13 GB RSS
[MEMORY] after chunk cleanup: 0.69 GB RSS

Chunk 2
[MEMORY] after reading chunk: 0.94 GB RSS
Loaded: 200,000 points
    Processing 2022-01-01 → 2024-12-31
      Using 88 observations (20220110 → 20241225)
    Processing 2020-01-01 → 2024-12-31
      Using 207 observations (20200103 → 20241225)
[MEMORY] after velocity calculation: 1.

In [12]:
# ============================================================
# CHECK TEMP PARQUET SCHEMAS
# ============================================================

for temp_file in temp_files:

    parquet_file = pq.ParquetFile(temp_file)

    columns = parquet_file.schema_arrow.names

    print(
        f"{temp_file.name}: "
        f"{len(columns)} columns"
    )

part_00001.parquet: 235 columns
part_00002.parquet: 235 columns
part_00003.parquet: 235 columns
part_00004.parquet: 231 columns
part_00005.parquet: 231 columns
part_00006.parquet: 231 columns
part_00007.parquet: 231 columns
part_00008.parquet: 231 columns


In [13]:
# ============================================================
# FIND COLUMN DIFFERENCES
# ============================================================

schemas = {}

for temp_file in temp_files:

    parquet_file = pq.ParquetFile(temp_file)

    schemas[temp_file.name] = set(
        parquet_file.schema_arrow.names
    )


all_columns = set().union(
    *schemas.values()
)

for filename, columns in schemas.items():

    missing = all_columns - columns

    if missing:
        print()
        print(filename)
        print("Missing columns:")
        print(sorted(missing))


part_00001.parquet
Missing columns:
['20210115', '20210214']

part_00002.parquet
Missing columns:
['20210115', '20210214']

part_00003.parquet
Missing columns:
['20210115', '20210214']

part_00004.parquet
Missing columns:
['20200327', '20200806', '20201210', '20201216', '20210906', '20211129']

part_00005.parquet
Missing columns:
['20200327', '20200806', '20201210', '20201216', '20210906', '20211129']

part_00006.parquet
Missing columns:
['20200327', '20200806', '20201210', '20201216', '20210906', '20211129']

part_00007.parquet
Missing columns:
['20200327', '20200806', '20201210', '20201216', '20210906', '20211129']

part_00008.parquet
Missing columns:
['20200327', '20200806', '20201210', '20201216', '20210906', '20211129']


In [14]:
# ============================================================
# DETERMINE UNION OF ALL COLUMNS
# ============================================================

all_columns = []

for temp_file in temp_files:

    parquet_file = pq.ParquetFile(temp_file)

    for column in parquet_file.schema_arrow.names:

        if column not in all_columns:
            all_columns.append(column)

print(
    f"Total columns in final dataset: "
    f"{len(all_columns)}"
)


Total columns in final dataset: 237


In [18]:
# ============================================================
# COMBINE ALL TEMPORARY PARQUETS
# ============================================================

print()
print("=" * 70)
print("COMBINING TEMPORARY PARQUETS")
print("=" * 70)

print(
    f"Temporary files: {len(temp_files)}"
)

print(
    f"Final output: {final_output}"
)


# ------------------------------------------------------------
# Establish final column order
# ------------------------------------------------------------

# Collect all unique columns from all temporary files

print(
    f"Total unique columns found: "
    f"{len(all_columns)}"
)


# ------------------------------------------------------------
# Identify time-series columns
# ------------------------------------------------------------

ts_columns = [
    column
    for column in all_columns
    if re.fullmatch(r"\d{8}", str(column))
]

# Sort chronologically
ts_columns = sorted(
    ts_columns,
    key=lambda x: pd.to_datetime(
        str(x),
        format="%Y%m%d"
    )
)


# ------------------------------------------------------------
# Identify velocity columns
# ------------------------------------------------------------

velocity_columns = [
    column
    for column in all_columns
    if str(column).startswith("VEL_")
    or str(column).startswith("V_STDERR_")
]


# ------------------------------------------------------------
# Identify geometry
# ------------------------------------------------------------

geometry_columns = [
    column
    for column in all_columns
    if column == "geometry"
]


# ------------------------------------------------------------
# Everything else = metadata
# ------------------------------------------------------------

special_columns = (
    set(ts_columns)
    | set(velocity_columns)
    | set(geometry_columns)
)

metadata_columns = [
    column
    for column in all_columns
    if column not in special_columns
]


# ------------------------------------------------------------
# Final column order
# ------------------------------------------------------------

ordered_columns = (
    metadata_columns
    + velocity_columns
    + ts_columns
    + geometry_columns
)


print()
print("=" * 70)
print("FINAL COLUMN ORDER")
print("=" * 70)

print(
    f"Metadata columns: "
    f"{len(metadata_columns)}"
)

print(
    f"Time-series columns: "
    f"{len(ts_columns)}"
)

print(
    f"Velocity columns: "
    f"{len(velocity_columns)}"
)

print(
    f"Geometry columns: "
    f"{len(geometry_columns)}"
)

print(
    f"Total columns: "
    f"{len(ordered_columns)}"
)


# ------------------------------------------------------------
# Show time-series range
# ------------------------------------------------------------

if ts_columns:

    print()
    print(
        f"Time series: "
        f"{ts_columns[0]} → {ts_columns[-1]}"
    )


# ------------------------------------------------------------
# Show velocity columns
# ------------------------------------------------------------

print()

print("Velocity columns:")

for column in velocity_columns:
    print(f"  {column}")


# ------------------------------------------------------------
# Use first file as base schema
# ------------------------------------------------------------

first_table = pq.read_table(
    temp_files[0]
)


# ------------------------------------------------------------
# Add missing columns to first table
# ------------------------------------------------------------

for column in ordered_columns:

    if column not in first_table.column_names:

        first_table = first_table.append_column(
            column,
            pa.nulls(
                first_table.num_rows,
                type=pa.float64()
            )
        )


# ------------------------------------------------------------
# Reorder first table
# ------------------------------------------------------------

first_table = first_table.select(
    ordered_columns
)

schema = first_table.schema

del first_table
gc.collect()


# ------------------------------------------------------------
# Create writer
# ------------------------------------------------------------

writer = pq.ParquetWriter(
    final_output,
    schema,
    compression="zstd"
)


# ------------------------------------------------------------
# Append all temporary Parquets
# ------------------------------------------------------------

for i, temp_file in enumerate(
    temp_files,
    start=1
):

    print()

    print(
        f"Combining "
        f"{i}/{len(temp_files)}: "
        f"{temp_file.name}"
    )

    table = pq.read_table(
        temp_file
    )


    # --------------------------------------------------------
    # Add missing columns
    # --------------------------------------------------------

    missing_columns = [
        column
        for column in ordered_columns
        if column not in table.column_names
    ]

    for column in missing_columns:

        field = schema.field(
            schema.get_field_index(column)
        )

        table = table.append_column(
            column,
            pa.nulls(
                table.num_rows,
                type=field.type
            )
        )


    # --------------------------------------------------------
    # Reorder columns
    # --------------------------------------------------------

    table = table.select(
        ordered_columns
    )


    # --------------------------------------------------------
    # Write
    # --------------------------------------------------------

    writer.write_table(
        table
    )


    del table
    gc.collect()


# ------------------------------------------------------------
# Close writer
# ------------------------------------------------------------

writer.close()


# ============================================================
# FINISHED
# ============================================================

print()

print("=" * 70)
print("FINAL PARQUET CREATED")
print("=" * 70)

print(final_output)

print_memory(
    "after final parquet"
)


COMBINING TEMPORARY PARQUETS
Temporary files: 8
Final output: /home/lukas/projects/ADUCAT/data/EGMS/vienna/processed/EGMS_022_2020_2024.parquet
Total unique columns found: 237

FINAL COLUMN ORDER
Metadata columns: 23
Time-series columns: 209
Velocity columns: 4
Geometry columns: 1
Total columns: 237

Time series: 20200103 → 20241225

Velocity columns:
  VEL_20220101_20241231
  V_STDERR_20220101_20241231
  VEL_20200101_20241231
  V_STDERR_20200101_20241231

Combining 1/8: part_00001.parquet

Combining 2/8: part_00002.parquet

Combining 3/8: part_00003.parquet

Combining 4/8: part_00004.parquet

Combining 5/8: part_00005.parquet

Combining 6/8: part_00006.parquet

Combining 7/8: part_00007.parquet

Combining 8/8: part_00008.parquet

FINAL PARQUET CREATED
/home/lukas/projects/ADUCAT/data/EGMS/vienna/processed/EGMS_022_2020_2024.parquet
[MEMORY] after final parquet: 1.48 GB RSS


In [19]:
combined_file = Path(f"{final_output}"
)
print(combined_file)

/home/lukas/projects/ADUCAT/data/EGMS/vienna/processed/EGMS_022_2020_2024.parquet


In [20]:
schema = pq.read_schema(combined_file)

print("=" * 70)
print("COMBINED PARQUET")
print("=" * 70)

print(f"Number of columns: {len(schema.names)}")

for i, column in enumerate(schema.names, start=1):
    print(f"{i:3d}: {column}")

COMBINED PARQUET
Number of columns: 237
  1: pid
  2: cluster_label
  3: mp_type
  4: easting
  5: northing
  6: height_ortho
  7: height_ellipse
  8: line
  9: pixel
 10: rmse_ts
 11: temporal_coherence
 12: amplitude_dispersion
 13: incidence_angle
 14: track_angle
 15: los_east
 16: los_north
 17: los_up
 18: mean_velocity
 19: mean_velocity_std
 20: acceleration
 21: acceleration_std
 22: seasonality
 23: seasonality_std
 24: VEL_20220101_20241231
 25: V_STDERR_20220101_20241231
 26: VEL_20200101_20241231
 27: V_STDERR_20200101_20241231
 28: 20200103
 29: 20200109
 30: 20200115
 31: 20200121
 32: 20200127
 33: 20200202
 34: 20200208
 35: 20200214
 36: 20200220
 37: 20200226
 38: 20200303
 39: 20200309
 40: 20200315
 41: 20200321
 42: 20200327
 43: 20200402
 44: 20200408
 45: 20200414
 46: 20200420
 47: 20200426
 48: 20200502
 49: 20200508
 50: 20200514
 51: 20200520
 52: 20200526
 53: 20200601
 54: 20200607
 55: 20200613
 56: 20200619
 57: 20200625
 58: 20200701
 59: 20200707
 60: 

### Seperate to velocity and timer series file

In [21]:
velocity_file = (
    OUTPUT_DIR /
    f"EGMS_{ORBIT}_{PRODUCT_RANGE}_without_timeseries.parquet"
)

timeseries_file = (
    OUTPUT_DIR /
    f"EGMS_{ORBIT}_{PRODUCT_RANGE}__just_timeseries.parquet"
)

print(velocity_file)
print(timeseries_file)

/home/lukas/projects/ADUCAT/data/EGMS/vienna/processed/EGMS_022_2020_2024_without_timeseries.parquet
/home/lukas/projects/ADUCAT/data/EGMS/vienna/processed/EGMS_022_2020_2024__just_timeseries.parquet


In [22]:
# --------------------------------------------------
# Full Parquet file
# --------------------------------------------------

pf = pq.ParquetFile(final_output)

columns = pf.schema_arrow.names

In [23]:
# --------------------------------------------------
# Identify time-series columns
# --------------------------------------------------

ts_cols = [
    c for c in columns
    if re.fullmatch(r"\d{8}", str(c))
]

# Everything else goes into spatial/velocity file
# --------------------------------------------------

velocity_columns = [
    c for c in columns
    if c not in ts_cols
]

print("Total columns:", len(columns))
print("Time-series columns:", len(ts_cols))
print("Velocity/static columns:", len(velocity_columns))

Total columns: 237
Time-series columns: 209
Velocity/static columns: 28


In [24]:
# ============================================================
# CREATE VELOCITY PARQUET
# ============================================================

print()
print("Creating velocity Parquet...")

velocity_writer = None

if velocity_file.exists():
    velocity_file.unlink()

for batch_number, batch in enumerate(
    pf.iter_batches(
        batch_size=CHUNK_SIZE,
        columns=velocity_columns
    ),
    start=1
):

    print(
        f"Velocity batch "
        f"{batch_number}: "
        f"{batch.num_rows:,} rows"
    )

    table = pa.Table.from_batches([batch])

    # --------------------------------------------------------
    # Create writer from FIRST batch only
    # --------------------------------------------------------

    if velocity_writer is None:

        velocity_writer = pq.ParquetWriter(
            velocity_file,
            table.schema,
            compression="zstd"
        )

        print(
            f"Created writer with "
            f"{len(table.schema.names)} columns"
        )

    # --------------------------------------------------------
    # Write batch
    # --------------------------------------------------------

    velocity_writer.write_table(table)

    del table
    del batch

    gc.collect()


# ------------------------------------------------------------
# Close writer
# ------------------------------------------------------------

if velocity_writer is not None:
    velocity_writer.close()
    velocity_writer = None

print(
    f"Velocity file created:\n"
    f"{velocity_file}"
)


Creating velocity Parquet...
Velocity batch 1: 200,000 rows
Created writer with 28 columns
Velocity batch 2: 200,000 rows
Velocity batch 3: 200,000 rows
Velocity batch 4: 200,000 rows
Velocity batch 5: 200,000 rows
Velocity batch 6: 200,000 rows
Velocity batch 7: 136,787 rows
Velocity file created:
/home/lukas/projects/ADUCAT/data/EGMS/vienna/processed/EGMS_022_2020_2024_without_timeseries.parquet


In [25]:
# ============================================================
# CREATE TIME-SERIES PARQUET
# ============================================================

print()
print("Creating time-series Parquet...")

timeseries_writer = None

if timeseries_file.exists():
    timeseries_file.unlink()

for batch_number, batch in enumerate(
    pf.iter_batches(
        batch_size=CHUNK_SIZE,
        columns=ts_cols
    ),
    start=1
):

    print(
        f"Time-series batch "
        f"{batch_number}: "
        f"{batch.num_rows:,} rows"
    )

    table = pa.Table.from_batches([batch])

    # --------------------------------------------------------
    # Create writer from FIRST batch only
    # --------------------------------------------------------

    if timeseries_writer is None:

        timeseries_writer = pq.ParquetWriter(
            timeseries_file,
            table.schema,
            compression="zstd"
        )

        print(
            f"Created writer with "
            f"{len(table.schema.names)} columns"
        )

    # --------------------------------------------------------
    # Write batch
    # --------------------------------------------------------

    timeseries_writer.write_table(table)

    del table
    del batch

    gc.collect()


# ------------------------------------------------------------
# Close writer
# ------------------------------------------------------------

if timeseries_writer is not None:
    timeseries_writer.close()
    timeseries_writer = None

print(
    f"Time-series file created:\n"
    f"{timeseries_file}"
)


Creating time-series Parquet...
Time-series batch 1: 200,000 rows
Created writer with 209 columns
Time-series batch 2: 200,000 rows
Time-series batch 3: 200,000 rows
Time-series batch 4: 200,000 rows
Time-series batch 5: 200,000 rows
Time-series batch 6: 200,000 rows
Time-series batch 7: 136,787 rows
Time-series file created:
/home/lukas/projects/ADUCAT/data/EGMS/vienna/processed/EGMS_022_2020_2024__just_timeseries.parquet


In [26]:
# ============================================================
# REMOVE TEMPORARY FILES
# ============================================================

if final_output.exists():

    print()
    print("Removing temporary files...")

    shutil.rmtree(temp_dir)

    print(
        f"Removed temporary directory: "
        f"{temp_dir}"
    )

else:

    print()
    print(
        "WARNING: Final output does not exist."
    )

    print(
        "Temporary files were NOT deleted."
    )


Removing temporary files...
Removed temporary directory: /home/lukas/projects/ADUCAT/data/EGMS/vienna/processed/temp_chunks
